# AI-Assisted Monthly Property Management Summary

This notebook creates a human-reviewed monthly property management summary using local, rule-based logic. It does **not** call any external API or use client data.

The output is saved to `reports/ai_assisted_monthly_summary.md` as a draft reporting aid for property managers and client stakeholders.

## 1. Project Purpose and Responsible AI Framing

The purpose of this notebook is to convert validated monthly portfolio KPIs into a professional draft narrative for a property management analyst workflow. The generated text is template-based and should be treated as a structured first draft, not as a final client communication.

Responsible use principles:

- Use only approved local project data.
- Avoid external API calls and client data use.
- Keep the output factual, traceable, and tied to the loaded monthly KPI tables.
- Require human review before any client delivery.
- Use the summary to accelerate reporting, not to replace analyst judgment.

## 2. Data Loading and Validation

This notebook uses exactly two input files:

- `data/ai_monthly_summary_inputs.csv`
- `data/processed/monthly_management_report_results.csv`

No original source CSVs are modified.

In [36]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

AI_INPUT_PATH = PROJECT_ROOT / "data" / "ai_monthly_summary_inputs.csv"
MONTHLY_REPORT_PATH = PROJECT_ROOT / "data" / "processed" / "monthly_management_report_results.csv"
OUTPUT_PATH = PROJECT_ROOT / "reports" / "ai_assisted_monthly_summary.md"

pd.set_option("display.max_columns", 100)
pd.options.display.float_format = "{:,.2f}".format

def dollars(value):
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value):,.0f}"

def dollars_m(value):
    sign = "-" if value < 0 else ""
    return f"{sign}${abs(value) / 1_000_000:,.1f}M"

def pct(value):
    return f"{value:.1%}"

def count(value):
    return f"{int(round(value)):,.0f}"

def direction_word(value, positive_good=True):
    if value > 0:
        return "improved" if positive_good else "increased"
    if value < 0:
        return "declined" if positive_good else "decreased"
    return "was flat"

for path in [AI_INPUT_PATH, MONTHLY_REPORT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")

ai_inputs = pd.read_csv(AI_INPUT_PATH, parse_dates=["month"])
monthly_report = pd.read_csv(MONTHLY_REPORT_PATH, parse_dates=["month"])

print(f"Loaded {AI_INPUT_PATH.relative_to(PROJECT_ROOT)}: {len(ai_inputs):,} rows, {len(ai_inputs.columns):,} columns")
print(f"Loaded {MONTHLY_REPORT_PATH.relative_to(PROJECT_ROOT)}: {len(monthly_report):,} rows, {len(monthly_report.columns):,} columns")

Loaded data/ai_monthly_summary_inputs.csv: 12 rows, 21 columns
Loaded data/processed/monthly_management_report_results.csv: 12 rows, 39 columns


In [37]:
required_ai_columns = {
    "month", "portfolio_budget_income", "portfolio_actual_income", "portfolio_budget_expenses",
    "portfolio_actual_expenses", "portfolio_noi", "portfolio_ebitda", "avg_occupancy",
    "total_ar_outstanding", "critical_ar_count", "high_ar_count", "invoice_count",
    "invoice_amount", "invoice_exception_count", "late_invoice_count", "work_order_count",
    "overdue_work_order_count", "avg_estimated_cost", "income_variance", "expense_variance",
    "ai_prompt_ready_note",
}

required_monthly_columns = {
    "month", "portfolio_budget_income", "portfolio_actual_income", "portfolio_income_variance",
    "portfolio_budget_expenses", "portfolio_actual_expenses", "portfolio_expense_variance",
    "portfolio_noi", "portfolio_ebitda", "average_occupancy_rate", "ar_outstanding_balance",
    "critical_ar_count", "high_ar_count", "tenants_with_ar_balance", "invoice_count",
    "invoice_amount", "invoice_exception_count", "late_invoice_count",
    "potential_duplicate_review_count", "missing_gl_code_count", "pending_invoice_count",
    "rejected_invoice_count", "work_order_count", "active_work_order_count",
    "overdue_work_order_count", "critical_work_order_count", "average_estimated_work_order_cost",
    "compliance_review_count", "open_compliance_finding_count", "open_high_severity_finding_count",
    "leasing_opportunity_count", "weighted_monthly_rent", "primary_management_focus",
}

missing_ai_columns = sorted(required_ai_columns - set(ai_inputs.columns))
missing_monthly_columns = sorted(required_monthly_columns - set(monthly_report.columns))
if missing_ai_columns or missing_monthly_columns:
    raise ValueError({
        "missing_ai_columns": missing_ai_columns,
        "missing_monthly_columns": missing_monthly_columns,
    })

validation_checks = []
validation_checks.append({"check": "AI input month values are unique", "passed": ai_inputs["month"].is_unique})
validation_checks.append({"check": "Monthly report month values are unique", "passed": monthly_report["month"].is_unique})
validation_checks.append({"check": "AI input has no null month values", "passed": ai_inputs["month"].notna().all()})
validation_checks.append({"check": "Monthly report has no null month values", "passed": monthly_report["month"].notna().all()})
validation_checks.append({"check": "Both files cover the same month set", "passed": set(ai_inputs["month"]) == set(monthly_report["month"])})

validation = pd.DataFrame(validation_checks)
display(validation)

if not validation["passed"].all():
    raise ValueError("Validation failed. Review the checks above before generating the summary.")

,check,passed
0,AI input month values are unique,True
1,Monthly report month values are unique,True
2,AI input has no null month values,True
3,Monthly report has no null month values,True
4,Both files cover the same month set,True


## 3. Latest-Month KPI Extraction

The latest reporting month is selected from the processed monthly management report. The prior month is retained for month-over-month context where available.

In [38]:
monthly_report = monthly_report.sort_values("month").reset_index(drop=True)
ai_inputs = ai_inputs.sort_values("month").reset_index(drop=True)

latest = monthly_report.iloc[-1]
prior = monthly_report.iloc[-2] if len(monthly_report) > 1 else None
latest_ai = ai_inputs.loc[ai_inputs["month"].eq(latest["month"])].iloc[0]

latest_kpis = pd.DataFrame([
    {"KPI": "Reporting Month", "Value": latest["month"].strftime("%B %Y")},
    {"KPI": "Portfolio NOI", "Value": dollars(latest["portfolio_noi"])},
    {"KPI": "Portfolio EBITDA", "Value": dollars(latest["portfolio_ebitda"])},
    {"KPI": "Average Occupancy", "Value": pct(latest["average_occupancy_rate"])},
    {"KPI": "Income Variance", "Value": dollars(latest["portfolio_income_variance"])},
    {"KPI": "Expense Variance", "Value": dollars(latest["portfolio_expense_variance"])},
    {"KPI": "AR Outstanding", "Value": dollars(latest["ar_outstanding_balance"])},
    {"KPI": "Critical / High AR Count", "Value": f"{count(latest['critical_ar_count'])} / {count(latest['high_ar_count'])}"},
    {"KPI": "Invoice Exceptions", "Value": count(latest["invoice_exception_count"])},
    {"KPI": "Overdue Work Orders", "Value": count(latest["overdue_work_order_count"])},
    {"KPI": "Open Compliance Findings", "Value": count(latest["open_compliance_finding_count"])},
    {"KPI": "Primary Management Focus", "Value": latest["primary_management_focus"]},
])

display(latest_kpis)

,KPI,Value
0,Reporting Month,December 2025
1,Portfolio NOI,"$29,900,399"
2,Portfolio EBITDA,"$28,155,519"
3,Average Occupancy,93.0%
4,Income Variance,"-$4,047,418"
5,Expense Variance,"$1,091,941"
6,AR Outstanding,"$941,879"
7,Critical / High AR Count,11 / 18
8,Invoice Exceptions,241
9,Overdue Work Orders,58


## 4. Executive Monthly Summary Generator

The generator below uses deterministic rules and templates. It reads KPI values, compares the latest month to the prior month, and produces a draft summary with financial, AR, vendor, work order, compliance, and follow-up sections.

In [39]:
def change(current, previous, column):
    if previous is None:
        return None
    return current[column] - previous[column]

def risk_level_from_counts(row):
    if row["open_high_severity_finding_count"] > 0:
        return "elevated due to open high-severity compliance findings"
    if row["critical_ar_count"] > 0:
        return "elevated due to critical AR exposure"
    if row["overdue_work_order_count"] >= 50:
        return "elevated due to work order SLA pressure"
    if row["invoice_exception_count"] >= 200:
        return "elevated due to vendor invoice exception volume"
    return "within routine monitoring thresholds"

def financial_risks(row, prior_row):
    risks = []
    if row["portfolio_income_variance"] < 0:
        risks.append(f"Actual income trailed budget by {dollars(abs(row['portfolio_income_variance']))}.")
    else:
        risks.append(f"Actual income exceeded budget by {dollars(row['portfolio_income_variance'])}.")

    if row["portfolio_expense_variance"] > 0:
        risks.append(f"Actual expenses exceeded budget by {dollars(row['portfolio_expense_variance'])}.")
    else:
        risks.append(f"Actual expenses were under budget by {dollars(abs(row['portfolio_expense_variance']))}.")

    if prior_row is not None:
        noi_change = change(row, prior_row, "portfolio_noi")
        ebitda_change = change(row, prior_row, "portfolio_ebitda")
        risks.append(f"NOI {direction_word(noi_change)} by {dollars(abs(noi_change))} month over month.")
        risks.append(f"EBITDA {direction_word(ebitda_change)} by {dollars(abs(ebitda_change))} month over month.")

    risks.append(f"Average occupancy was {pct(row['average_occupancy_rate'])}.")
    return risks

def ar_priorities(row, prior_row):
    priorities = [
        f"AR outstanding was {dollars(row['ar_outstanding_balance'])}.",
        f"The reporting month included {count(row['critical_ar_count'])} critical and {count(row['high_ar_count'])} high-priority AR records.",
        f"{count(row['tenants_with_ar_balance'])} tenants had an AR balance in the reporting month.",
    ]
    if prior_row is not None:
        ar_change = change(row, prior_row, "ar_outstanding_balance")
        priorities.append(f"AR outstanding {direction_word(ar_change, positive_good=False)} by {dollars(abs(ar_change))} versus the prior month.")
    return priorities

def vendor_notes(row, prior_row):
    exception_rate = row["invoice_exception_count"] / row["invoice_count"] if row["invoice_count"] else 0
    notes = [
        f"The portfolio processed {count(row['invoice_count'])} invoices totaling {dollars(row['invoice_amount'])}.",
        f"Invoice exceptions totaled {count(row['invoice_exception_count'])}, or {exception_rate:.1%} of invoice volume.",
        f"Late invoices totaled {count(row['late_invoice_count'])}; potential duplicate review cases totaled {count(row['potential_duplicate_review_count'])}; missing GL code items totaled {count(row['missing_gl_code_count'])}.",
        f"Pending approvals totaled {count(row['pending_invoice_count'])}, and rejected invoices totaled {count(row['rejected_invoice_count'])}.",
    ]
    if prior_row is not None:
        exception_change = change(row, prior_row, "invoice_exception_count")
        notes.append(f"Invoice exceptions {direction_word(exception_change, positive_good=False)} by {count(abs(exception_change))} versus the prior month.")
    return notes

def work_order_compliance_concerns(row, prior_row):
    concerns = [
        f"Work order volume totaled {count(row['work_order_count'])}, with {count(row['active_work_order_count'])} active work orders and {count(row['overdue_work_order_count'])} overdue work orders.",
        f"Critical work orders totaled {count(row['critical_work_order_count'])}; average estimated cost was {dollars(row['average_estimated_work_order_cost'])}.",
        f"Compliance reviews totaled {count(row['compliance_review_count'])}; open compliance findings totaled {count(row['open_compliance_finding_count'])}, including {count(row['open_high_severity_finding_count'])} high-severity open findings.",
    ]
    if prior_row is not None:
        overdue_change = change(row, prior_row, "overdue_work_order_count")
        open_compliance_change = change(row, prior_row, "open_compliance_finding_count")
        concerns.append(f"Overdue work orders {direction_word(overdue_change, positive_good=False)} by {count(abs(overdue_change))} month over month.")
        concerns.append(f"Open compliance findings {direction_word(open_compliance_change, positive_good=False)} by {count(abs(open_compliance_change))} month over month.")
    return concerns

def follow_up_questions(row):
    questions = [
        "Which properties drove the negative income variance and expense overage this month?",
        "Which tenants make up the critical and high-priority AR balances, and what collection actions are already underway?",
        "Which vendors or invoice categories account for the largest invoice exception volume?",
        "Do potential duplicate invoice review cases have supporting documentation and approval history before payment?",
        "Which overdue work orders are tenant-impacting, critical, or blocked by vendor availability?",
        "Which open high-severity compliance findings require client escalation before the next reporting cycle?",
    ]
    if row["primary_management_focus"]:
        questions.insert(0, f"What actions are planned for the stated primary management focus: {row['primary_management_focus']}?")
    return questions

def bullet_list(items):
    return "\n".join(f"- {item}" for item in items)

def generate_monthly_summary(row, prior_row, ai_row):
    report_month = row["month"].strftime("%B %Y")
    risk_level = risk_level_from_counts(row)
    note = "The source row is prepared for a human-reviewed monthly property management summary."

    summary = f"""# AI-Assisted Monthly Property Management Summary - {report_month}

## Responsible AI Disclosure

This summary was generated locally using a rule-based reporting template and validated project CSV inputs. No external API or client data was used. The narrative is intended as a draft reporting aid and must be reviewed by a property management analyst before any client-facing delivery.

Source note: {note}

## Executive Monthly Summary

For {report_month}, the portfolio reported NOI of {dollars(row['portfolio_noi'])} and EBITDA of {dollars(row['portfolio_ebitda'])}. Average occupancy was {pct(row['average_occupancy_rate'])}. The primary management focus for the month is **{row['primary_management_focus']}**, and overall operating risk is {risk_level}.

## Latest-Month KPI Snapshot

| KPI | Value |
|---|---:|
| Portfolio budget income | {dollars(row['portfolio_budget_income'])} |
| Portfolio actual income | {dollars(row['portfolio_actual_income'])} |
| Income variance | {dollars(row['portfolio_income_variance'])} |
| Portfolio budget expenses | {dollars(row['portfolio_budget_expenses'])} |
| Portfolio actual expenses | {dollars(row['portfolio_actual_expenses'])} |
| Expense variance | {dollars(row['portfolio_expense_variance'])} |
| AR outstanding | {dollars(row['ar_outstanding_balance'])} |
| Invoice exceptions | {count(row['invoice_exception_count'])} |
| Overdue work orders | {count(row['overdue_work_order_count'])} |
| Open compliance findings | {count(row['open_compliance_finding_count'])} |

## Top Financial Risks

{bullet_list(financial_risks(row, prior_row))}

## AR Follow-Up Priorities

{bullet_list(ar_priorities(row, prior_row))}

## Vendor Invoice Exception Review Notes

{bullet_list(vendor_notes(row, prior_row))}

## Work Order and Compliance Concerns

{bullet_list(work_order_compliance_concerns(row, prior_row))}

## Recommended Follow-Up Questions for Property Managers

{bullet_list(follow_up_questions(row))}

## Human-Review Disclosure

This summary was generated locally using a rule-based reporting template and validated project CSV inputs. No external API or client data was used. The narrative is intended as a draft reporting aid and must be reviewed by a property management analyst before any client-facing delivery.
"""
    return summary


## 5. Top Financial Risks

The financial risk section focuses on budget variance, NOI, EBITDA, and occupancy for the latest reporting month.

In [40]:
financial_risk_table = pd.DataFrame({"Financial Risk Note": financial_risks(latest, prior)})
display(financial_risk_table)

,Financial Risk Note
0,"Actual income trailed budget by $4,047,418."
1,"Actual expenses exceeded budget by $1,091,941."
2,"NOI declined by $197,953 month over month."
3,"EBITDA declined by $347,230 month over month."
4,Average occupancy was 93.0%.


## 6. AR Follow-Up Priorities

The AR section highlights aggregate exposure and the number of records requiring high or critical follow-up.

In [41]:
ar_priority_table = pd.DataFrame({"AR Follow-Up Priority": ar_priorities(latest, prior)})
display(ar_priority_table)

,AR Follow-Up Priority
0,"AR outstanding was $941,879."
1,The reporting month included 11 critical and 1...
2,"2,145 tenants had an AR balance in the reporti..."
3,"AR outstanding decreased by $156,255 versus th..."


## 7. Vendor Invoice Exception Review Notes

The vendor invoice section summarizes exception volume and control-related review items. Potential duplicate review cases are treated as review flags, not confirmed duplicate invoices.

In [42]:
vendor_note_table = pd.DataFrame({"Vendor Invoice Review Note": vendor_notes(latest, prior)})
display(vendor_note_table)

,Vendor Invoice Review Note
0,The portfolio processed 885 invoices totaling ...
1,"Invoice exceptions totaled 241, or 27.2% of in..."
2,Late invoices totaled 103; potential duplicate...
3,"Pending approvals totaled 102, and rejected in..."
4,Invoice exceptions increased by 4 versus the p...


## 8. Work Order and Compliance Concerns

The operations and compliance section focuses on SLA risk, active work order volume, overdue work orders, open findings, and high-severity compliance items.

In [43]:
operations_concern_table = pd.DataFrame({"Work Order and Compliance Concern": work_order_compliance_concerns(latest, prior)})
display(operations_concern_table)

,Work Order and Compliance Concern
0,"Work order volume totaled 314, with 66 active ..."
1,Critical work orders totaled 15; average estim...
2,Compliance reviews totaled 174; open complianc...
3,Overdue work orders decreased by 5 month over ...
4,Open compliance findings increased by 8 month ...


## 9. Recommended Follow-Up Questions for Property Managers

These questions are intended to guide the human review process and help property managers validate the story behind the KPIs.

In [44]:
follow_up_question_table = pd.DataFrame({"Recommended Follow-Up Question": follow_up_questions(latest)})
display(follow_up_question_table)

,Recommended Follow-Up Question
0,What actions are planned for the stated primar...
1,Which properties drove the negative income var...
2,Which tenants make up the critical and high-pr...
3,Which vendors or invoice categories account fo...
4,Do potential duplicate invoice review cases ha...
5,Which overdue work orders are tenant-impacting...
6,Which open high-severity compliance findings r...


## 10. Generate and Save the Human-Reviewed Draft

The final cell generates the draft monthly summary and saves it as markdown for review.

In [45]:
monthly_summary = generate_monthly_summary(latest, prior, latest_ai)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(monthly_summary, encoding="utf-8")

print(f"Saved monthly summary to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
display(Markdown(monthly_summary))

Saved monthly summary to reports/ai_assisted_monthly_summary.md


# AI-Assisted Monthly Property Management Summary - December 2025

## Responsible AI Disclosure

This summary was generated locally using a rule-based reporting template and validated project CSV inputs. No external API or client data was used. The narrative is intended as a draft reporting aid and must be reviewed by a property management analyst before any client-facing delivery.

Source note: The source row is prepared for a human-reviewed monthly property management summary.

## Executive Monthly Summary

For December 2025, the portfolio reported NOI of $29,900,399 and EBITDA of $28,155,519. Average occupancy was 93.0%. The primary management focus for the month is **Compliance and control follow-up**, and overall operating risk is elevated due to open high-severity compliance findings.

## Latest-Month KPI Snapshot

| KPI | Value |
|---|---:|
| Portfolio budget income | $52,346,601 |
| Portfolio actual income | $48,299,183 |
| Income variance | -$4,047,418 |
| Portfolio budget expenses | $17,306,843 |
| Portfolio actual expenses | $18,398,784 |
| Expense variance | $1,091,941 |
| AR outstanding | $941,879 |
| Invoice exceptions | 241 |
| Overdue work orders | 58 |
| Open compliance findings | 98 |

## Top Financial Risks

- Actual income trailed budget by $4,047,418.
- Actual expenses exceeded budget by $1,091,941.
- NOI declined by $197,953 month over month.
- EBITDA declined by $347,230 month over month.
- Average occupancy was 93.0%.

## AR Follow-Up Priorities

- AR outstanding was $941,879.
- The reporting month included 11 critical and 18 high-priority AR records.
- 2,145 tenants had an AR balance in the reporting month.
- AR outstanding decreased by $156,255 versus the prior month.

## Vendor Invoice Exception Review Notes

- The portfolio processed 885 invoices totaling $2,987,883.
- Invoice exceptions totaled 241, or 27.2% of invoice volume.
- Late invoices totaled 103; potential duplicate review cases totaled 11; missing GL code items totaled 33.
- Pending approvals totaled 102, and rejected invoices totaled 24.
- Invoice exceptions increased by 4 versus the prior month.

## Work Order and Compliance Concerns

- Work order volume totaled 314, with 66 active work orders and 58 overdue work orders.
- Critical work orders totaled 15; average estimated cost was $618.
- Compliance reviews totaled 174; open compliance findings totaled 98, including 31 high-severity open findings.
- Overdue work orders decreased by 5 month over month.
- Open compliance findings increased by 8 month over month.

## Recommended Follow-Up Questions for Property Managers

- What actions are planned for the stated primary management focus: Compliance and control follow-up?
- Which properties drove the negative income variance and expense overage this month?
- Which tenants make up the critical and high-priority AR balances, and what collection actions are already underway?
- Which vendors or invoice categories account for the largest invoice exception volume?
- Do potential duplicate invoice review cases have supporting documentation and approval history before payment?
- Which overdue work orders are tenant-impacting, critical, or blocked by vendor availability?
- Which open high-severity compliance findings require client escalation before the next reporting cycle?

## Human-Review Disclosure

This summary was generated locally using a rule-based reporting template and validated project CSV inputs. No external API or client data was used. The narrative is intended as a draft reporting aid and must be reviewed by a property management analyst before any client-facing delivery.
